# Big-O and Complexity


```{contents}
:local:
:depth: 2
```

A finance team can scan every expense once or compare every pair. Before timing either program, define the input and count the work. Analysis describes a cost model; measurement checks how that model behaves on a particular system.

## Choose an Input and a Cost Model

```{index} Choose an Input and a Cost Model
```


Let `n` be the number of expense records. Count one addition when a record contributes to a total. Array access, integer addition, and comparison are treated as constant-cost operations here. String comparisons, database requests, and growing collection operations need their own assumptions.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0, 1, 4, 8 })
{
    int[] expenses = Enumerable.Repeat(125, n).ToArray();
    long total = 0, additions = 0;
    foreach (int cents in expenses) { total += cents; additions++; }
    Console.WriteLine($"n={n}, total={total}, additions={additions}");
}

The selected operation runs exactly `n` times, including zero times for an empty input. Allocation and initialization in this demonstration are setup; include them when analyzing the entire program. The sum uses `long` to hold many integer amounts.

## Sequential, Rectangular, and Triangular Loops

```{index} Sequential, Rectangular, and Triangular Loops
```


Add costs of sequential phases. Multiply independent loop ranges. For dependent ranges, sum the inner counts instead of multiplying their largest bounds blindly.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0, 1, 4, 8 })
{
    long sequential = 0, square = 0, triangle = 0;
    for (int i = 0; i < n; i++) sequential++;
    for (int i = 0; i < n; i++) sequential++;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++) square++;
    for (int i = 0; i < n; i++)
        for (int j = i + 1; j < n; j++) triangle++;
    Console.WriteLine($"n={n}: sequential={sequential}, square={square}, distinct pairs={triangle}");
}

The counts are `2n`, `n²`, and `n(n−1)/2`. The triangular loop compares each unordered pair once, excluding self-pairs. Both pair scans are Θ(n²), despite different constants. A loop inside another loop can still be linear if its inner range is bounded by a fixed constant.

## Logarithmic Loops and Multiple Inputs

```{index} Logarithmic Loops and Multiple Inputs
```


Integer halving performs exactly floor(log₂ n) reductions for positive `n`. Zero performs none. Distinguish reductions from the number of states visited. If there are `n` invoices and `m` vendors, the comparison count is Θ(nm) when both sizes are positive; do not replace `m` with `n` without a reason.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0, 1, 2, 3, 8, 9, 16 })
{
    int reductions = 0;
    for (int size = n; size > 1; size /= 2) reductions++;
    Console.WriteLine($"n={n}, reductions={reductions}");
}
int invoices = 3, vendors = 5, comparisons = 0;
for (int i = 0; i < invoices; i++)
    for (int j = 0; j < vendors; j++) comparisons++;
Console.WriteLine($"invoice/vendor comparisons={comparisons}");

For asymptotic statements, restrict to sufficiently large positive `n`, where logarithms are defined. A doubling loop has logarithmic iterations too, but an unguarded integer update can overflow; repeated division avoids that problem in this example.

## Bounds and Tight Growth

```{index} Bounds and Tight Growth
```


For nonnegative costs, `T(n) ∈ O(f(n))` means there are constants `c > 0` and `n₀` such that `T(n) ≤ c f(n)` for all `n ≥ n₀`. Ω supplies a lower bound; Θ supplies both bounds. These symbols describe a chosen cost function, not automatically a worst-case cost.

In [ ]:
using System;
using System.Linq;

for (long n = 1; n <= 5; n++)
{
    long cost = 3 * n * n + 5 * n + 20;
    Console.WriteLine($"n={n}, 3n^2={3*n*n} <= {cost} <= 28n^2={28*n*n}");
}

For `n ≥ 1`, `3n² ≤ 3n² + 5n + 20 ≤ 28n²`, so the cost is Θ(n²). It is also O(n³), an upper bound that loses useful precision. Saying O(n²) alone does not establish that the cost is quadratic: a linear cost also satisfies that upper bound. Logarithm bases differ by constant factors.

## Best, Worst, Expected, and Amortized Costs

```{index} Best, Worst, Expected, and Amortized Costs
```


A search can stop early. Best and worst cases vary over inputs of the same size; an expected cost needs a probability model. An amortized cost averages over a sequence of operations and does not require random inputs.

In [ ]:
using System;
using System.Linq;

int[] ids = { 40, 10, 30, 20 };
foreach (int target in new[] { 40, 20, 99 })
{
    int checks = 0, found = -1;
    for (int i = 0; i < ids.Length; i++)
    {
        checks++;
        if (ids[i] == target) { found = i; break; }
    }
    Console.WriteLine($"target={target}, found={found}, checks={checks}");
}

On nonempty arrays, this search has best-case Θ(1) and worst-case Θ(n) comparisons. If a successful target is equally likely at every index, its expected count is `(n+1)/2`, hence Θ(n). A geometric-growth array append has amortized Θ(1) cost even though one resizing append costs Θ(n). Expected hashing requires assumptions about key distribution and table policy; it is a different claim.

## Space, Growth, and Measurement

```{index} Space, Growth, and Measurement
```


State whether you count total space or auxiliary space. Auxiliary space excludes the input but includes temporary arrays, returned storage unless explicitly excluded, and active recursion frames. Peak live memory is different from all allocations over a run.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 8, 16, 32 })
{
    double log = Math.Log2(n);
    Console.WriteLine($"n={n}: log={log:F0}, n={n}, nlogn={n*log:F0}, n^2={(long)n*n}, 2^n={Math.Pow(2,n):F0}");
}
int[] amounts = { 100, 250, 75 };
long sum = 0;
foreach (int value in amounts) sum += value;
int[] copy = amounts.ToArray();
Console.WriteLine($"sum={sum}, copied items={copy.Length}");

The sum uses Θ(1) auxiliary space; copying uses Θ(n). A method may have low stack depth and still allocate a large result. Timings depend on JIT compilation, garbage collection, I/O, data, and hardware. Warm up, repeat, report input sizes and conditions, and compare relevant operation counts. A single stopwatch result cannot establish an asymptotic bound.

## Exercise

```{index} Analysis practice
```

Count a fixed-width inner loop for n=0,4,8: each outer iteration does exactly three checks. Explain why nesting alone does not imply quadratic cost.

In [ ]:
using System;

// Exercise
// Count a fixed-width inner loop for n=0,4,8: each outer iteration does exactly three checks. Explain why nesting alone does not imply quadratic cost.
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;

// Solution
foreach(int n in new[]{0,4,8})
{
 long checks=0;
 for(int i=0;i<n;i++)for(int j=0;j<3;j++)checks++;
 Console.WriteLine($"n={n}, checks={checks}");
}
// Exact 3n checks; Theta(n) for growing n.

n=0, checks=0
n=4, checks=12
n=8, checks=24


```{rubric} Footnotes
```
[^1]: The fixed-width arithmetic cost model is suitable for these C# examples; arbitrary-precision arithmetic and variable-length strings require additional size parameters.
[^2]: Asymptotic bounds hold beyond a threshold. Constants and input distributions still matter when selecting a practical implementation.